# Try TabPFN, gradient boosting and the rest on SlaveVoyages

This notebook builds the database, then compares models on one target. **Runtime > Change runtime type > GPU** (a free T4 is enough).

You need: the data file `tastdb-exp-2019.csv` (see DATA.md in the repo), and a free TabPFN key from Prior Labs (<https://priorlabs.ai/tabpfn-3-5>; accept the model license on their site).

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/slavevoyages-project.git"   # <- put the real address here
!git clone -q {REPO_URL} repo
%cd repo
!pip install -q -r requirements.txt -r requirements-tabpfn.txt

## 2. Add the data and your TabPFN key

Upload `tastdb-exp-2019.csv` with the file icon on the left (or run the cell below). For the key: click the key icon on the left, add a secret named exactly `TABPFN_TOKEN`, and switch on *Notebook access*.

In [ ]:
from google.colab import files, userdata
import os
if not os.path.exists('tastdb-exp-2019.csv'):
    files.upload()                                   # choose tastdb-exp-2019.csv
os.environ['TABPFN_TOKEN'] = userdata.get('TABPFN_TOKEN')

## 3. Build the database and check everything works

In [ ]:
!python voyage_db.py sqlite tastdb-exp-2019.csv voyages.sqlite
!python tests/test_framework.py

## 4. Compare models

`mode='real'` uses only recorded values; `mode='imputed'` fills gaps with the authors' imputed values. `train_frac` is the share of *known* values used for training.

In [ ]:
import models                                       # registers 'tabpfn' because it is installed
from run_experiment import run
print(sorted(models.REGISTRY))
res, task = run('NATIONAL_GROUP', mode='real', scheme='random', train_frac=0.5,
                model_names=['majority_or_median', 'tree', 'gradient_boosting', 'random_forest', 'tabpfn'],
                csv='voyages.sqlite')
res[['model', 'accuracy', 'macro_f1', 'balanced_acc', 'n_train', 'settings', 'seconds']].round(3)

## 5. Try other settings

Time-based split (train on early voyages, test on later ones), and a number target:

In [ ]:
res_t, _ = run('NATIONAL_GROUP', 'real', 'temporal', 0.6, model_names=['majority_or_median', 'tree', 'gradient_boosting', 'tabpfn'], csv='voyages.sqlite')
print(res_t[['model', 'accuracy', 'macro_f1']].round(3).to_string(index=False))

res_n, _ = run('TONNAGE', 'real', 'random', 0.5, model_names=['majority_or_median', 'tree', 'gradient_boosting', 'tabpfn'], csv='voyages.sqlite')
print(res_n[['model', 'MAE_tons', 'median_abs_err_tons']].round(1).to_string(index=False))

## Add your own model

Write one function and register it. It receives the task (`X_train`, `y_train`, `X_val`, `y_val`, `X_test`) and returns predictions for `X_test`:

```python
from models import register
@register('my_model', kinds=('classification',))
def my_model(task):
    ...
    return predictions
```